# FinReq AI — Fine-Tuning Qwen2.5 on Google Colab (No Google Drive Needed)

This notebook fine-tunes **Qwen2.5-7B-Instruct** (or 3B fallback) using **Unsloth + QLoRA** completely inside Colab session storage.
**No Google Drive connection required.** You simply upload `train.jsonl` and `val.jsonl` directly into Colab.

### 1. Upload `train.jsonl` and `val.jsonl` (Directly from PC)

In [ ]:
# Run this cell and select `train.jsonl` and `val.jsonl` from your computer:
# Local path: c:\Users\vikas\Desktop\projectfinal\ml\dataset\data\
# (Or simply drag and drop the files into the Files folder on the left panel)
from google.colab import files
import os

if not os.path.exists('train.jsonl') or not os.path.exists('val.jsonl'):
    print('Please select train.jsonl and val.jsonl to upload:')
    uploaded = files.upload()
else:
    print('train.jsonl and val.jsonl are already uploaded in /content!')

!ls -lh *.jsonl

### 2. Install Unsloth and Training Dependencies

In [ ]:
%%capture
!pip install --no-deps unsloth
!pip install "torch" "transformers" "datasets" "accelerate" "bitsandbytes" "trl" "peft" "xformers"


### 3. Load Base Model in 4-bit (Runs on Free T4 GPU)

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048  # Keeps memory low on free Colab
DTYPE = None          # None for auto-detection
LOAD_IN_4BIT = True

# Primary: unsloth/Qwen2.5-7B-Instruct-bnb-4bit (or fallback: unsloth/Qwen2.5-3B-Instruct-bnb-4bit)
MODEL_NAME = 'unsloth/Qwen2.5-7B-Instruct-bnb-4bit'

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)
print('Base model loaded successfully into GPU memory.')

### 4. Configure QLoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
print('LoRA adapter configured.')

### 5. Format Dataset with Qwen Chat Template

In [ ]:
from datasets import load_dataset

dataset = load_dataset('json', data_files={'train': 'train.jsonl', 'val': 'val.jsonl'})

def formatting_prompts_func(examples):
    convos = examples['messages']
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return {'text': texts}

dataset = dataset.map(formatting_prompts_func, batched=True)
print(f'Loaded {len(dataset["train"])} training examples and {len(dataset["val"])} validation examples.')

### 6. Train Model (Loss on Assistant Responses Only)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

OUTPUT_DIR = '/content/outputs'

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset['train'],
    eval_dataset=dataset['val'],
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        warmup_ratio=0.03,
        num_train_epochs=2,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        eval_strategy='steps',
        eval_steps=50,
        save_strategy='steps',
        save_steps=100,
        optim='paged_adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='cosine',
        seed=42,
        output_dir=OUTPUT_DIR,
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part='<|im_start|>user\n',
    response_part='<|im_start|>assistant\n',
)

print('Starting training...')
trainer_stats = trainer.train()
print('Training complete!')

### 7. Export GGUF Q4_K_M (for Local Ollama CPU Inference)

In [ ]:
# Export to GGUF Q4_K_M
model.save_pretrained_gguf('/content', tokenizer, quantization_method='q4_k_m')
print('Exported to GGUF in /content!')
!ls -lh /content/*.gguf

### 8. Download GGUF Directly to Your Computer

In [ ]:
import glob
from google.colab import files

gguf_files = glob.glob('/content/*q4_k_m*.gguf') or glob.glob('/content/*.gguf')
if gguf_files:
    target = gguf_files[0]
    print(f'Downloading {target} directly to your PC...')
    files.download(target)
else:
    print('GGUF file not found in /content. Check files list:')
    !ls -lh /content